# ENN583 — Week 6 Practical: Pose Graph Optimisation with SymForce

Visual odometry usually estimates motion one frame pair at a time. Small errors therefore accumulate into **drift**. In this practical we select keyframes, estimate motion over several overlapping keyframe baselines, and use a pose graph to find the trajectory that best agrees with all of those measurements.

By the end, you should be able to:

- represent camera poses and relative-pose measurements with `Pose3`;
- write a between-pose residual on the pose manifold;
- combine consecutive and non-consecutive local motion estimates;
- construct factors and choose which values SymForce should optimise; and
- adapt the example to refine a visual-odometry trajectory.

> The example is synthetic so that we know the correct answer. The poses, factors, and optimiser are the same ones we would use for real VO measurements.

## 0. Setup

SymForce is included in the supplied ENN583 environment. If it is missing from a different environment, install it once with `pip install symforce`, then restart the kernel.

SymForce must be told how to handle small numerical singularities **before** its symbolic module is imported.

In [ ]:
import symforce

symforce.set_epsilon_to_symbol()

import matplotlib.pyplot as plt
import numpy as np
import symforce.symbolic as sf
from spatialmath import SE3
from symforce.opt.factor import Factor
from symforce.opt.optimizer import Optimizer
from symforce.values import Values

SEED = 583
rng = np.random.default_rng(SEED)

## 1. Poses and relative-pose measurements

We write `world_T_camera` for the pose that transforms a point from the camera frame into the world frame. If two world poses are `world_T_a` and `world_T_b`, their predicted relative motion is

$$ {}^aT_b = ({}^wT_a)^{-1}\,{}^wT_b. $$

The first pose below is at the world origin. The remaining poses trace a short curved trajectory. `Pose3` represents all six degrees of freedom, even though this example stays close to the ground plane.

In [ ]:
def make_pose(x, z, heading):
    """Make a 3D pose for a camera moving on the x-z ground plane."""
    rotation = sf.Rot3.from_yaw_pitch_roll(0.0, -heading, 0.0)
    translation = sf.V3(x, 0.0, z)
    return sf.Pose3(rotation, translation)


NUM_KEYFRAMES = 100
KEYFRAME_STRIDE = 5
frame_indices = np.arange(NUM_KEYFRAMES) * KEYFRAME_STRIDE

# A smooth 20 m reference path for the selected keyframes.
distance = np.linspace(0.0, 20.0, NUM_KEYFRAMES)
x = distance
z = 1.5 * np.sin(0.35 * distance) + 0.04 * distance
heading = np.arctan2(np.gradient(z), np.gradient(x))

true_poses = [
    make_pose(x_i, z_i, heading_i)
    for x_i, z_i, heading_i in zip(x, z, heading)
]

print(f"Selected {NUM_KEYFRAMES} keyframes from {frame_indices[-1] + 1} images")

### Converting between SpatialMath and SymForce

In earlier practicals we used SpatialMath's `SE3` class to store rigid transforms. SymForce uses `Pose3` for the same rotation-and-translation information. The two libraries have different Python types, but both can be converted through a 4×4 homogeneous transformation matrix.

These functions do not invert or change the direction of a transform. An `SE3` representing `a_T_b` becomes a `Pose3` representing the same `a_T_b`.

In [ ]:
def spatialmath_to_symforce(spatial_pose):
    """Convert a SpatialMath SE3 into a SymForce Pose3."""
    T = spatial_pose.A
    rotation = sf.Rot3.from_rotation_matrix(sf.Matrix33(T[:3, :3]))
    translation = sf.V3(*T[:3, 3])
    return sf.Pose3(rotation, translation)


def symforce_to_spatialmath(symforce_pose):
    """Convert a SymForce Pose3 into a SpatialMath SE3."""
    rotation = np.array(
        symforce_pose.R.to_rotation_matrix(), dtype=float
    ).reshape(3, 3)
    translation = np.array(symforce_pose.t, dtype=float).reshape(3)
    return SE3.Rt(rotation, translation)

In [ ]:
# Check that converting in both directions preserves the transform.
spatial_pose = SE3.Trans(1.0, 0.0, 2.0) * SE3.Ry(0.2)
symforce_pose = spatialmath_to_symforce(spatial_pose)
spatial_pose_recovered = symforce_to_spatialmath(symforce_pose)

print("SpatialMath SE3 matrix:\n", spatial_pose.A)
print("\nSymForce Pose3:\n", symforce_pose)
print("\nRound trip successful:", np.allclose(spatial_pose.A, spatial_pose_recovered.A))

A keyframe VO system gives us noisy relative motions, not the true world poses. Here we simulate one measurement between every pair of consecutive **keyframes**. Each keyframe is five image frames after the previous one. We then accumulate the measurements exactly as we accumulated motion in Week 5.

The noise vector has six entries: three for rotation and three for translation in the local tangent space of `Pose3`.

In [ ]:
odometry_measurements = []

for pose_a, pose_b in zip(true_poses[:-1], true_poses[1:]):
    true_relative_pose = pose_a.inverse() * pose_b

    # Consecutive VO has both random error and a small systematic drift.
    noise = rng.normal(
        loc=[0.0, 0.0, 0.002, 0.008, 0.0, 0.004],
        scale=[0.004, 0.004, 0.006, 0.025, 0.005, 0.025],
    )
    # retract() applies this small 6D tangent-space perturbation to the pose
    # while keeping the result a valid Pose3. The first three values perturb
    # rotation and the final three perturb translation.
    noisy_relative_pose = true_relative_pose.retract(sf.V6(*noise), sf.numeric_epsilon)
    odometry_measurements.append(noisy_relative_pose)

# Accumulating the noisy motions gives the initial VO trajectory.
initial_poses = [true_poses[0]]
for relative_pose in odometry_measurements:
    initial_poses.append(initial_poses[-1] * relative_pose)

### Before refinement: evaluate the accumulated VO trajectory

Before adding any extra constraints, compare the accumulated keyframe VO trajectory with ground truth. We report two translation metrics:

- **translation RMSE:** error over the complete trajectory;
- **final-position error:** the accumulated drift at the final keyframe.

In [ ]:
def plot_trajectory(poses, label, style):
    x = [float(pose.t[0]) for pose in poses]
    z = [float(pose.t[2]) for pose in poses]
    plt.plot(x, z, style, label=label)


def translation_rmse(estimated, reference):
    squared_errors = []
    for estimate, truth in zip(estimated, reference):
        difference = np.array(estimate.t, dtype=float) - np.array(truth.t, dtype=float)
        squared_errors.append(difference @ difference)
    return np.sqrt(np.mean(squared_errors))


def final_position_error(estimated, reference):
    difference = (
        np.array(estimated[-1].t, dtype=float)
        - np.array(reference[-1].t, dtype=float)
    )
    return np.linalg.norm(difference)


plt.figure(figsize=(7, 6))
plot_trajectory(true_poses, "Ground truth", "k--")
plot_trajectory(initial_poses, "Accumulated keyframe VO", "C0-")
plt.axis("equal")
plt.xlabel("world x (m)")
plt.ylabel("world z (m)")
plt.title("Before refinement: accumulated keyframe VO")
plt.grid()
plt.legend();

print("Before pose-graph optimisation")
print(f"  Translation RMSE:    {translation_rmse(initial_poses, true_poses):.3f} m")
print(f"  Final-position error: {final_position_error(initial_poses, true_poses):.3f} m")

## 2. Estimate motion over a longer baseline

Consecutive keyframes are not the only pairs with shared visual content. If keyframes $i$ and $i+2$ still overlap, we can match their features and estimate ${}^iT_{i+2}$ **directly**, rather than obtaining it by multiplying ${}^iT_{i+1}$ and ${}^{i+1}T_{i+2}$.

The direct estimate and the accumulated estimate will differ because every motion estimate is noisy. Their disagreement creates redundancy that the pose graph can use to smooth the trajectory. Here each keyframe is also connected to keyframes two, three, and four steps ahead, as long as they remain in the local matching window.

In [ ]:
additional_edges = [
    (i, i + gap)
    for gap in (2, 3, 4)
    for i in range(len(true_poses) - gap)
]
additional_measurements = []

for i, j in additional_edges:
    true_relative_pose = true_poses[i].inverse() * true_poses[j]
    noise = rng.normal(
        loc=0.0,
        scale=[0.003, 0.003, 0.004, 0.012, 0.004, 0.012],
    )
    measurement = true_relative_pose.retract(sf.V6(*noise), sf.numeric_epsilon)
    additional_measurements.append(measurement)

print(f"{len(additional_edges)} additional local measurements")

### Visualise a subset of the VO measurements

The plot below shows the first 12 keyframes. A measurement from keyframe $i$ predicts where keyframe $j$ should be when composed with the current pose estimate $X_i$.

The consecutive measurements end exactly at the next accumulated VO pose because those measurements were used to construct the initial trajectory. The independently estimated longer-baseline measurements end nearby, but not at exactly the same location. These disagreements are the errors that the pose graph will reconcile.

In [ ]:
NUM_POSES_TO_SHOW = 12

plt.figure(figsize=(10, 5))
plot_trajectory(
    initial_poses[:NUM_POSES_TO_SHOW],
    "Accumulated keyframe poses",
    "ko-",
)

# Consecutive measurements used to build the initial trajectory.
for i in range(NUM_POSES_TO_SHOW - 1):
    predicted_pose = initial_poses[i] * odometry_measurements[i]
    plt.plot(
        [float(initial_poses[i].t[0]), float(predicted_pose.t[0])],
        [float(initial_poses[i].t[2]), float(predicted_pose.t[2])],
        "C0-",
        linewidth=2,
        label="Consecutive measurement" if i == 0 else None,
    )

# Independent measurements to other keyframes in the local window.
measurement_was_labelled = False
for measurement, (i, j) in zip(additional_measurements, additional_edges):
    if j >= NUM_POSES_TO_SHOW:
        continue
    predicted_pose = initial_poses[i] * measurement
    plt.plot(
        [float(initial_poses[i].t[0]), float(predicted_pose.t[0])],
        [float(initial_poses[i].t[2]), float(predicted_pose.t[2])],
        "C3-",
        alpha=0.65,
        label="Longer-baseline measurement" if not measurement_was_labelled else None,
    )
    plt.plot(float(predicted_pose.t[0]), float(predicted_pose.t[2]), "C3x")
    measurement_was_labelled = True

plt.axis("equal")
plt.xlabel("world x (m)")
plt.ylabel("world z (m)")
plt.title("VO measurements for the first 12 keyframes")
plt.grid()
plt.legend();

## 3. Define the between-pose factor

For each edge, the graph compares the measured relative pose with the relative pose predicted by the two current pose estimates. Ordinary subtraction is not meaningful for rotations, so `local_coordinates` computes their six-dimensional difference on the pose manifold.

Each component is divided by its expected standard deviation. A precise measurement therefore has a larger `sqrt_information` value and contributes more strongly to the objective. SymForce differentiates this function for us; we do not write a Jacobian.

In [ ]:
def between_residual(
    pose_a: sf.Pose3,
    pose_b: sf.Pose3,
    measured_a_T_b: sf.Pose3,
    sqrt_information: sf.V6,
    epsilon: sf.Scalar,
) -> sf.V6:
    predicted_a_T_b = pose_a.inverse() * pose_b
    # local_coordinates() expresses the difference from the measured pose to
    # the predicted pose as a 6D tangent-space vector. It is zero when the
    # poses agree and avoids incorrectly subtracting rotation matrices.
    pose_error = measured_a_T_b.local_coordinates(predicted_a_T_b, epsilon)
    return sf.V6(*pose_error).multiply_elementwise(sqrt_information)

## 4. Store the graph values

A `Values` object holds both kinds of quantities in our factor graph:

- **variables:** the world poses that we want to refine;
- **fixed inputs:** the relative-pose measurements, their weights, and epsilon.

The wider-baseline measurements are more precise in this example: more camera translation gives better parallax and a better-conditioned motion estimate. This is not guaranteed—matching also becomes harder as the baseline grows. A real keyframe system would keep only geometrically verified pairs and could set uncertainty from the inlier geometry or motion-fit residuals.

In [ ]:
values = Values(
    poses=initial_poses,
    odometry=odometry_measurements,
    additional_motion=additional_measurements,
    odometry_sqrt_information=sf.V6(
        1 / 0.008, 1 / 0.008, 1 / 0.01, 1 / 0.04, 1 / 0.01, 1 / 0.04
    ),
    additional_sqrt_information=sf.V6(
        1 / 0.005, 1 / 0.005, 1 / 0.007, 1 / 0.02, 1 / 0.008, 1 / 0.02
    ),
    epsilon=sf.numeric_epsilon,
)

## 5. Build the factors

A factor connects keys in `Values` to a residual function. We first add one odometry factor for every consecutive keyframe pair. We then add factors for the other measurements in the local keyframe window.

Notice that the graph structure is explicit in the key strings.

In [ ]:
factors = []

for i in range(len(odometry_measurements)):
    factors.append(
        Factor(
            residual=between_residual,
            keys=[
                f"poses[{i}]",
                f"poses[{i + 1}]",
                f"odometry[{i}]",
                "odometry_sqrt_information",
                "epsilon",
            ],
        )
    )

for measurement_index, (i, j) in enumerate(additional_edges):
    factors.append(
        Factor(
            residual=between_residual,
            keys=[
                f"poses[{i}]",
                f"poses[{j}]",
                f"additional_motion[{measurement_index}]",
                "additional_sqrt_information",
                "epsilon",
            ],
        )
    )

print(f"{len(initial_poses)} pose variables")
print(f"{len(factors)} relative-pose factors")

## 6. Optimise the graph

Relative measurements alone do not define an absolute world frame: moving every pose by the same transform would leave every residual unchanged. We remove this **gauge freedom** by holding `poses[0]` fixed and optimising only poses 1 onward.

In [ ]:
keys_to_optimise = [f"poses[{i}]" for i in range(1, len(initial_poses))]

optimizer = Optimizer(
    factors=factors,
    optimized_keys=keys_to_optimise,
    params=Optimizer.Params(verbose=False),
)
result = optimizer.optimize(values)
optimised_poses = result.optimized_values["poses"]

print("Status:", result.status)
print(f"Final squared error: {result.error():.3f}")

In [ ]:
plt.figure(figsize=(7, 6))
plot_trajectory(true_poses, "Ground truth", "k--")
plot_trajectory(initial_poses, "Accumulated keyframe VO", "C0-")
plot_trajectory(optimised_poses, "Pose graph", "C1-")
# Draw a subset of the local factors so the plot remains readable.
for edge_number, (i, j) in enumerate(additional_edges[::10]):
    plt.plot(
        [float(optimised_poses[i].t[0]), float(optimised_poses[j].t[0])],
        [float(optimised_poses[i].t[2]), float(optimised_poses[j].t[2])],
        "r:",
        alpha=0.5,
        label="Example local factors" if edge_number == 0 else None,
    )
plt.axis("equal")
plt.xlabel("world x (m)")
plt.ylabel("world z (m)")
plt.title("Pose-graph refinement")
plt.grid()
plt.legend();

In [ ]:
print("After pose-graph optimisation")
print(f"  Translation RMSE:    {translation_rmse(optimised_poses, true_poses):.3f} m")
print(f"  Final-position error: {final_position_error(optimised_poses, true_poses):.3f} m")

### Check your understanding

1. Why did the consecutive odometry factors alone not change the accumulated VO trajectory?
2. What would happen if `poses[0]` were also included in `keys_to_optimise`?
3. Increase the standard deviations of `additional_sqrt_information`. How and why does the result change?
4. Extend the local window to keyframe $i+5$. Does it improve the result? What eventually limits how far apart matched keyframes can be?

## 7. Use your own visual-odometry result

The Week 5 notebooks store each relative motion as a SpatialMath `SE3` in the **camera coordinate frame**, where $Z$ points forward. KITTI ground-truth poses use the IMU coordinate frame, where $X$ points forward. Before constructing the graph, change the basis of every relative motion exactly as we did for the Week 5 trajectory plot:

```python
motion_in_imu = T_cam_imu.inv() @ motion_in_camera @ T_cam_imu
```

This changes the coordinate basis without changing the direction of the transform. The Week 5 lists already contain ${}^{k}T_{k+1}$ in the direction needed for trajectory accumulation. The 2D–2D notebook calls its list `relative_motions`; the PnP notebook calls it `relative_motion`.

The Week 5 processing loop always estimates motion between `frame` and `frame + 1`. For this practical, modify that code so it can instead estimate relative motion between any chosen pair `frame_a` and `frame_b`. You will need this for direct measurements between non-consecutive frames. The feature matching, 2D–2D or PnP motion estimation, scale recovery, and transform direction remain the same.

You do not need to introduce keyframes in your own solution. Use every image frame as a pose variable. The existing consecutive-frame motions provide the initial trajectory and the consecutive graph edges.

In [ ]:
# Example handover from the Week 5 2D-2D notebook:
# frame_motions_in_imu = [
#     T_cam_imu.inv() @ motion @ T_cam_imu
#     for motion in relative_motions
# ]
# odometry_measurements = [
#     spatialmath_to_symforce(motion)
#     for motion in frame_motions_in_imu
# ]
#
# initial_poses = [sf.Pose3.identity()]
# for relative_pose in odometry_measurements:
#     initial_poses.append(initial_poses[-1] * relative_pose)
#
# For comparison with KITTI ground truth:
# true_poses = [
#     spatialmath_to_symforce(data.ground_truth_pose(frame))
#     for frame in range(data.frame_count)
# ]

### Additional non-consecutive frame factors

The existing Week 5 motions provide the consecutive frame edges and the initial trajectory.

For an additional graph edge such as $(i, i+2)$, use your modified Week 5 code to estimate motion **directly** between image frames `i` and `i + 2`. Change that measurement from the camera basis into the IMU basis, convert it to `Pose3`, and store the graph edge using the frame indices `(i, i + 2)`. Repeat for other overlapping frame pairs such as $(i, i+3)$ while the images still share enough visual content.


## Take-away

A pose graph does not create new information: it makes all available relative-pose measurements agree as well as possible. Consecutive keyframe edges provide the initial trajectory. Direct estimates to other overlapping keyframes in a local window add redundancy and allow the optimiser to smooth inconsistent motion estimates. Measurement direction, uncertainty, a good initial guess, and a fixed reference pose are all essential.